## Censo 2020 ageb

https://www.inegi.org.mx/programas/ccpv/2020/#datos_abiertos (Principales resultados por AGEB y manzana urbana)

In [ ]:
import zipfile

path_zip = "../data/raw/ageb_mza_urbana_31_cpv2020_csv.zip"

with zipfile.ZipFile(path_zip, 'r') as z:
    for filename in z.namelist():
        if filename.endswith('.csv'):
            print(filename)

In [ ]:
import pandas as pd
import zipfile

path_zip = "../data/raw/ageb_mza_urbana_31_cpv2020_csv.zip"

with zipfile.ZipFile(path_zip, 'r') as z:
    # Busca automáticamente el archivo del conjunto de datos dentro del ZIP
    csv_target = [f for f in z.namelist() if f.endswith('.csv') and 'conjunto_de_datos' in f][0]
    
    with z.open(csv_target) as f:
        df_censo = pd.read_csv(f, encoding='utf-8', low_memory=False)

df_censo.head()

In [ ]:
df_censo.columns.tolist()

In [ ]:
[c for c in df_censo.columns if "AGEB" in c.upper()]

In [ ]:
[c for c in df_censo.columns if "POB" in c.upper()]

In [ ]:
# Filtrar registros de la ciudad de Mérida
censo_merida = df_censo[
    (df_censo["MUN"] == 50) &
    (df_censo["LOC"] == 1)
].copy()

print("Registros de Mérida:", len(censo_merida))

censo_merida[["MUN", "NOM_MUN", "LOC", "NOM_LOC", "AGEB", "MZA"]].head(10)

In [ ]:
censo_ageb = censo_merida[
    (censo_merida["NOM_LOC"] == "Total AGEB urbana") &
    (censo_merida["MZA"] == 0)
].copy()

print("AGEB en Censo:", len(censo_ageb))
print("AGEB únicos:", censo_ageb["AGEB"].nunique())

censo_ageb[
    ["MUN", "NOM_MUN", "LOC", "AGEB", "POBTOT", "POB0_14", "POB15_64", "POB65_MAS", "PEA"]
].head()

In [ ]:
ageb_cartografia = set(ageb_merida_ciudad["CVE_AGEB"])
ageb_censo = set(censo_ageb["AGEB"])

print("AGEB en cartografía:", len(ageb_cartografia))
print("AGEB en Censo:", len(ageb_censo))

print("AGEB en ambos:", len(ageb_cartografia & ageb_censo))
print("AGEB solo en cartografía:", len(ageb_cartografia - ageb_censo))
print("AGEB solo en Censo:", len(ageb_censo - ageb_cartografia))

## Marco Geoestadístico. Censo de Población y Vivienda 2020

http://inegi.org.mx/app/biblioteca/ficha.html?upc=889463807469

In [ ]:
%pip install geopandas

In [ ]:
import geopandas as gpd

print(gpd.__version__)

In [ ]:
import zipfile

path_zip = "../data/raw/31_yucatan.zip"

with zipfile.ZipFile(path_zip, 'r') as z:
    shapefiles = [f for f in z.namelist() if f.endswith('.shp')]

for shp in shapefiles:
    print(shp)

In [ ]:
import os
import pandas as pd
import geopandas as gpd

results = []

for shp in shapefiles:

    path_mapa = f"zip://{path_zip}!{shp}"
    gdf = gpd.read_file(path_mapa)

    results.append({
        "file": os.path.basename(shp),
        "rows": len(gdf),
        "columns": len(gdf.columns),
        "crs": str(gdf.crs),
        "geometry": gdf.geometry.geom_type.unique().tolist(),
        "valid_geometries": gdf.geometry.is_valid.sum()
    })

layers_info = pd.DataFrame(results)
layers_info

In [ ]:
import geopandas as gpd

gdf_31ti = gpd.read_file(
    "zip://../data/raw/31_yucatan.zip!conjunto_de_datos/31ti.shp"
)

gdf_31ti.head()

In [ ]:
gdf_31ti.columns.tolist()

In [ ]:
gdf_31ti.crs

In [ ]:
gdf_31ti.geometry.geom_type.value_counts()

In [ ]:
gdf_31ti.geometry.is_valid.value_counts()

In [ ]:
import geopandas as gpd

path_zip = "../data/raw/31_yucatan.zip"

gdf_31m = gpd.read_file(f"zip://{path_zip}!conjunto_de_datos/31m.shp")
gdf_31a = gpd.read_file(f"zip://{path_zip}!conjunto_de_datos/31a.shp")

print("31m")
print(gdf_31m.columns.tolist())
print(gdf_31m.head())

print("\n31a")
print(gdf_31a.columns.tolist())
print(gdf_31a.head())

In [ ]:
print("31m CRS:", gdf_31m.crs)
print("31m geometrías:", gdf_31m.geometry.geom_type.value_counts())
print("31m válidas:", gdf_31m.geometry.is_valid.value_counts())

print("\n31a CRS:", gdf_31a.crs)
print("31a geometrías:", gdf_31a.geometry.geom_type.value_counts())
print("31a válidas:", gdf_31a.geometry.is_valid.value_counts())

In [ ]:
gdf_31a["CVE_MUN"].value_counts()

In [ ]:
ageb_merida["CVE_AGEB"].nunique()

In [ ]:
# Municipios disponibles en la capa 31a
print("Municipios:")
print(gdf_31a["CVE_MUN"].value_counts().sort_index())

# Verificar Mérida
print("\n¿Existe CVE_MUN = 050?")
print((gdf_31a["CVE_MUN"] == "050").any())

# Filtrar Mérida
ageb_merida = gdf_31a[
    gdf_31a["CVE_MUN"] == "050"
].copy()

print("\nNúmero de polígonos en Mérida:")
print(len(ageb_merida))

print("\nNúmero de AGEB diferentes:")
print(ageb_merida["CVE_AGEB"].nunique())

print("\nLocalidades dentro de Mérida:")
print(ageb_merida["CVE_LOC"].value_counts())

print("\nPrimeros registros:")
display(ageb_merida.head())

In [ ]:
import matplotlib.pyplot as plt

ax = ageb_merida.plot(
    figsize=(10, 10),
    edgecolor="black",
    linewidth=0.3
)

ax.set_title("AGEB del municipio de Mérida")
ax.set_axis_off()

plt.show()

In [ ]:
ageb_merida_ciudad = gdf_31a[
    (gdf_31a["CVE_MUN"] == "050") &
    (gdf_31a["CVE_LOC"] == "0001")
].copy()

print("AGEB de la ciudad de Mérida:", len(ageb_merida_ciudad))

In [ ]:
manzanas_merida = gdf_31m[
    (gdf_31m["CVE_MUN"] == "050") &
    (gdf_31m["CVE_LOC"] == "0001")
].copy()

print("Manzanas en Mérida:", len(manzanas_merida))

In [ ]:
manzanas_merida["AMBITO"].value_counts()

In [ ]:
ageb_urbanas = set(ageb_merida_ciudad["CVE_AGEB"])

ageb_manzanas = set(
    manzanas_merida["CVE_AGEB"]
)

print("AGEB en la cartografía:", len(ageb_urbanas))
print("AGEB con manzanas:", len(ageb_manzanas))

print(
    "AGEB sin manzanas:",
    len(ageb_urbanas - ageb_manzanas)
)

In [ ]:
import matplotlib.pyplot as plt

ax = ageb_merida_ciudad.plot(
    figsize=(10, 10),
    edgecolor="black",
    linewidth=0.3
)

ax.set_title("AGEB de la ciudad de Mérida")
ax.set_axis_off()

plt.show()

In [ ]:
print(ageb_merida_ciudad.crs)
print(ageb_merida_ciudad.geometry.geom_type.unique())
print(ageb_merida_ciudad.geometry.is_valid.all())
print(ageb_merida_ciudad.shape)
print(ageb_merida_ciudad["CVE_AGEB"].nunique())


## DENUE

https://www.inegi.org.mx/app/descarga/default.html 

In [ ]:
import zipfile

zip_path = "../data/raw/denue_31_csv.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    print("Archivos dentro del ZIP:")
    for file in z.namelist():
        print(file)

In [ ]:
import pandas as pd
import zipfile

with zipfile.ZipFile("../data/raw/denue_31_csv.zip") as z:
    with z.open("conjunto_de_datos/denue_inegi_31_.csv") as f:
        denue_sample = pd.read_csv(f, nrows=5, encoding="latin-1")

print("Columns:", len(denue_sample.columns))
print(denue_sample.columns.tolist())

In [ ]:
with zipfile.ZipFile("../data/raw/denue_31_csv.zip") as z:
    with z.open("conjunto_de_datos/denue_inegi_31_.csv") as f:
        denue_df = pd.read_csv(
            f,
            encoding="latin-1",
            low_memory=False
        )

print("Records:", len(denue_df))
print("Columns:", len(denue_df.columns))

In [ ]:
print(denue_df[["cve_mun", "municipio", "cve_loc", "localidad"]].head())
print(denue_df[["cve_mun", "cve_loc"]].dtypes)

In [ ]:
denue_merida = denue_df[
    (denue_df["cve_mun"] == 50) &
    (denue_df["cve_loc"] == 1)
].copy()

print("Establishments in Merida:", len(denue_merida))
print("Unique AGEBs:", denue_merida["ageb"].nunique())

denue_merida[
    ["id", "nom_estab", "codigo_act", "nombre_act",
     "ageb", "manzana", "latitud", "longitud"]
].head()

In [ ]:
ageb_denue = set(denue_merida["ageb"].astype(str).str.zfill(4))
ageb_cartography = set(ageb_merida_ciudad["CVE_AGEB"].astype(str).str.zfill(4))

print("AGEBs in cartography:", len(ageb_cartography))
print("AGEBs in DENUE:", len(ageb_denue))
print("AGEBs in both:", len(ageb_cartography & ageb_denue))
print("AGEBs only in cartography:", len(ageb_cartography - ageb_denue))
print("AGEBs only in DENUE:", len(ageb_denue - ageb_cartography))

print("\nAGEB only in cartography:")
print(ageb_cartography - ageb_denue)

In [ ]:
print("AGEBs only in DENUE:")
print(sorted(ageb_denue - ageb_cartography))

print("\nAGEBs only in cartography:")
print(sorted(ageb_cartography - ageb_denue))

In [ ]:
denue_missing_ageb = denue_merida[
    denue_merida["ageb"].astype(str).str.zfill(4).isin(
        ageb_denue - ageb_cartography
    )
]

denue_missing_ageb[
    [
        "ageb",
        "nom_estab",
        "nombre_act",
        "municipio",
        "localidad",
        "latitud",
        "longitud"
    ]
].head(20)

In [ ]:
print(
    denue_missing_ageb["ageb"]
    .astype(str)
    .str.zfill(4)
    .value_counts()
    .sort_index()
)

In [ ]:
import os

for root, dirs, files in os.walk("../data/raw"):
    for file in files:
        print(os.path.join(root, file))

## ATUS

http://inegi.org.mx/programas/accidentes/#datos_abiertos 

In [ ]:
import zipfile
import os
import pandas as pd
import geopandas as gpd

path_zip = "../data/raw/atus_2024_shp.zip"

with zipfile.ZipFile(path_zip, "r") as z:
    shapefiles = [f for f in z.namelist() if f.endswith(".shp")]

shapefiles

In [ ]:
path_mapa = f"zip://{path_zip}!{shapefiles[0]}"

gdf = gpd.read_file(path_mapa)

gdf.head()

In [ ]:
gdf.columns.tolist()

In [ ]:
print(gdf.crs)
print(gdf.geometry.geom_type.unique())

In [ ]:
gdf.shape

In [ ]:
import zipfile
import geopandas as gpd

path_zip = "../data/raw/atus_2024_shp.zip"

with zipfile.ZipFile(path_zip, "r") as z:
    shapefiles = [f for f in z.namelist() if f.endswith(".shp")]

path_mapa = f"zip://{path_zip}!{shapefiles[0]}"

gdf = gpd.read_file(path_mapa)

print(f"Registros extraídos: {len(gdf)}")
print(f"Columnas: {len(gdf.columns)}")
print(f"CRS: {gdf.crs}")

In [ ]:
gdf[["LATITUD", "LONGITUD", "geometry"]].head()

In [ ]:
gdf[["EDO", "MPIO", "TIPACCID", "ANIO", "MES", "DIA", "HORA"]].head()

In [ ]:
print(ageb_merida.crs)
print(ageb_merida.geometry.geom_type.unique())
print(ageb_merida.geometry.is_valid.all())
print(ageb_merida.shape)
print(ageb_merida["CVE_AGEB"].nunique())

In [ ]:
atus_ageb = gdf.to_crs(ageb_merida.crs)

print(atus_ageb.crs)

In [ ]:
atus_ageb = gpd.sjoin(
    atus_ageb,
    ageb_merida[["CVE_AGEB", "geometry"]],
    how="left",
    predicate="within"
)

print(atus_ageb.shape)
print(atus_ageb["CVE_AGEB"].notna().sum())

In [ ]:
print(gdf["EDO"].value_counts().head())
print(gdf["EDO"].unique())

In [ ]:
print(gdf[["LATITUD", "LONGITUD"]].describe())

In [ ]:
atus_yucatan = gdf[gdf["EDO"] == 31]

print(atus_yucatan["MPIO"].value_counts().sort_index())

In [ ]:
atus_yucatan[["EDO", "MPIO"]].drop_duplicates().sort_values("MPIO")

In [ ]:
atus_merida = gdf[
    (gdf["EDO"] == 31) &
    (gdf["MPIO"] == 50)
].copy()

print("Accidentes en Mérida:", len(atus_merida))

In [ ]:
atus_merida = atus_merida.to_crs(ageb_merida.crs)

atus_merida_ageb = gpd.sjoin(
    atus_merida,
    ageb_merida[["CVE_AGEB", "geometry"]],
    how="left",
    predicate="within"
)

print("Accidentes:", len(atus_merida_ageb))
print("Accidentes asignados a AGEB:", atus_merida_ageb["CVE_AGEB"].notna().sum())

In [ ]:
print("Total de accidentes:", len(atus_merida_ageb))
print("Asignados a AGEB:", atus_merida_ageb["CVE_AGEB"].notna().sum())
print("Sin AGEB:", atus_merida_ageb["CVE_AGEB"].isna().sum())
print(
    "Porcentaje asignado:",
    round(atus_merida_ageb["CVE_AGEB"].notna().mean() * 100, 2),
    "%"
)